# Prompt-Variants + Ollama V1-Notebook.

In [48]:
'''Objective: 
Develop a small prototype (or Jupyter Notebook) that takes sample resume bullet points 
(provided below) and rewrites each one into improved bullet points using 3-4 different 
LLMs or different prompting techniques.  '''

'Objective: \nDevelop a small prototype (or Jupyter Notebook) that takes sample resume bullet points \n(provided below) and rewrites each one into improved bullet points using 3-4 different \nLLMs or different prompting techniques.  '

In [49]:
# Below code will have option to use both :- Different Prompting Techniques and Different LLM Models.
# Since I donot have enough money to invest on OpenAI, Anthropic and Gemini APIs. And free APIs of them have very limited number of tokens.
# I have used open source LLMs from OLLAMA
# V2 version of this code will take input directly from user by pasting all bullet points. 
# Apart from that you can paste your bullet points in question variable within inverted commas. 

In [50]:
# Make sure you have installed Ollama and all relevant models used in below code.
# After that type "ollama serve"  in power shell so that it can run locally in computer.
# Next type ".venv/Scripts/activate" in powershell after correctly locating the folder in powershell.
# Next type "python -m pip install -U jupyterlab" in powershell to install jupyter notebook in folder
# To run jupyter notebook type "python -m jupyter lab" in powershell.   

In [51]:
# Below code Installs charset detection lib to reduce RequestsDependencyWarning.

In [52]:
! pip install charset-normalizer  

In [53]:
# Installs the Python "ollama" package (note: not required if you use OpenAI client to hit Ollama’s OpenAI-compatible API)

In [54]:
! pip install ollama  

In [55]:
# imports  # Section label to group imports.

import os  # Built-in: OS utilities (paths, env vars, etc.).
import requests  # HTTP client for quick endpoint checks (like localhost:11434).
from openai import OpenAI  # OpenAI Python SDK client (also works with Ollama OpenAI-compatible endpoints).
from dotenv import load_dotenv  # Loads environment variables from a .env file (used when you have API_KEYS in .env file and you are using OpenAI APIs).
from IPython.display import Markdown, display, update_display  # Notebook display helpers (update_display unused here).

In [56]:
requests.get("http://localhost:11434").content  # Pings Ollama server root to confirm it’s running and returns raw response bytes.

b'Ollama is running'

In [57]:
# constants  # Section label for configuration/constants.
# MODEL_GPT = 'gpt-4o-mini'  # Example OpenAI model (commented out; not used).

OLLAMA_BASE_URL = "http://localhost:11434/v1"  # Ollama’s OpenAI-compatible base URL (must end with /v1).


# Below is the list of all open source model list and coressponding tags. Which will be useful in later part

MODEL_GPTOSS = 'gpt-oss:120b-cloud'  # GPT OSS 120 Billion Parameters- Cloud Variant tag.
MODEL_GEMMA4 = 'gemma3:4b'  # Gemma 3 4B tag as per your local Ollama pulls.
MODEL_GEMMA = 'gemma3:1b'  # Gemma 3 1B tag (lighter/faster).
MODEL_DEEPSEEK = 'deepseek-r1:1.5b'  # DeepSeek R1 distilled/small tag.
MODEL_LLAMA = 'llama3.2'  # Llama 3.2 tag.

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")  # Creates OpenAI SDK client pointed at Ollama; api_key is a dummy value for Ollama.
# api_key is mandatory input in OpenAI() Hence written dummy key.

In [58]:
# Multiline input bullets you want rewritten.

question = """  
• Integrated the Google Form Transaction page with the website which lead to 2 min payment process, from 5min 
• Designed and deployed the marketing web page section of the site.  
• Developed and organized the Github repository to streamline workflow. 
"""  # End of user bullet input text.

In [59]:
# System rules telling the model how to rewrite bullets.

system_prompt = """  
• Start with a strong action verb (e.g., Analyzed, Led, Designed, Developed, etc.). 
• Include relevant industry keywords (e.g. for sales, marketing, software engineering).
• Mention measurable achievements by adding genuine numerical figures, if they’re reasonable from the original content (e.g. percentage increase, time saved).
• Be concise but informative (aim for ~12–20 words). Only give 4 bullent points.
• Maintain professional and clear language.
• Make sentences more impact driven. And avoid generic fluff.
• Make bullet points ATS friendly.
"""  # End system instruction block.

user_prompt = "For each input bullet point, generate equal number of rewritten bullet points:" + question  # Builds a user prompt 

In [60]:
def get_prompt_versions(user_prompt, system_prompt):  # Creates multiple prompting styles (zero-shot, few-shot, etc.).

    zero_shot = [  # Zero-shot: only system + user instruction.
        {"role": "system", "content": system_prompt},  # System rules.
        {"role": "user", "content": user_prompt}  # User content to rewrite.
    ]  # End zero-shot message list.

    few_shot = [  # Few-shot: give one example pair before the actual question.
        {"role": "system", "content": system_prompt},  # System rules.
        {"role": "user", "content":  # Example prompt message (input + output in one message).
         "Input: •Managed team meetings.\n"
         "Output: •Led team coordination improving meeting efficiency by 30%.\n"},  # Example demonstration.
        {"role": "user", "content": user_prompt}  # Actual user question.
    ]  # End few-shot message list.

    chain_of_thought = [  # CoT style: asks model to reason step-by-step (often not ideal for smaller models).
        {"role": "system", "content": system_prompt},  # System rules.
        {"role": "user", "content":  # User asks to think step-by-step.
         "Think step by step and rewrite professionally:\n\n" + user_prompt}  # CoT instruction + content.
    ]  # End chain-of-thought message list.

    role_play = [  # Role-play: adds persona in an extra system message.
        {"role": "system", "content":  # Persona system message.
         "You are a senior technical recruiter."},  # Sets role.
        {"role": "system", "content": system_prompt},  # Your formatting rules as system prompt.
        {"role": "user", "content": user_prompt}  # User content to rewrite.
    ]  # End role-play message list.

    return {  # Returns a dict so you can pick prompt style by name.
        "Zero-Shot": zero_shot,  # Dict entry for zero-shot.
        "Few-Shot": few_shot,  # Dict entry for few-shot.
        "Chain-of-Thought": chain_of_thought,  # Dict entry for CoT.
        "Role-Play (Recruiter)": role_play  # Dict entry for recruiter persona.
    }  # End dict return.


# ---------- Pretty Printer (for clarity) ----------  # Visual separator comment.
def pretty_print_messages(messages):  # Prints messages so you can verify structure.
    for i, msg in enumerate(messages, start=1):  # Loops through message list with 1-based index.
        print(f"\n--- MESSAGE {i} ---")  # Prints message number header.
        print("ROLE   :", msg["role"])  # Prints role field.
        print("CONTENT:")  # Prints label.
        print(msg["content"])  # Prints content text.

In [61]:
prompts = get_prompt_versions(user_prompt, system_prompt)  # Builds all prompt variants from your question + system prompt.

In [62]:
# Print all clearly  # Comment describing the loop intent.
for name, messages in prompts.items():  # Iterates each prompt variant.
    print(f"\n================ {name.upper()} ================")  # Prints section header per prompt style.
    pretty_print_messages(messages)  # Prints the actual structured messages.

# Below results gives clear picture on how different prompting style be taken by different models.


================ ZERO-SHOT ================

--- MESSAGE 1 ---
ROLE   : system
CONTENT:
  
• Start with a strong action verb (e.g., Analyzed, Led, Designed, Developed, etc.). 
• Include relevant industry keywords (e.g. for sales, marketing, software engineering).
• Mention measurable achievements by adding genuine numerical figures, if they’re reasonable from the original content (e.g. percentage increase, time saved).
• Be concise but informative (aim for ~12–20 words). Only give 4 bullent points.
• Maintain professional and clear language.
• Make sentences more impact driven. And avoid generic fluff.
• Make bullet points ATS friendly.


--- MESSAGE 2 ---
ROLE   : user
CONTENT:
For each input bullet point, generate equal number of rewritten bullet points:  
• Integrated the Google Form Transaction page with the website which lead to 2 min payment process, from 5min 
• Designed and deployed the marketing web page section of the site.  
• Developed and organized the Github repository t

In [63]:
#messages = [  # Old attempt to build messages manually (commented out).
#    {"role": "system", "content": system_prompt},  # System instruction.
#    {"role": "system", "content": user_prompt}  
#]  # End commented list.

In [73]:
# Get gptoss to answer  # Comment describing which model you’re running.

response = ollama.chat.completions.create(  # Calls Ollama’s OpenAI-compatible /chat/completions endpoint.
    model=MODEL_GPTOSS,  # Chooses model tag.
    messages=prompts["Chain-of-Thought"]  # Sends CoT prompt variant.
)  # End API call.

reply = response.choices[0].message.content  # Extracts assistant message text from first choice.

display(Markdown(reply))  # Renders output as Markdown in the notebook.

- Integrated Google Form transaction page, slashing payment time from 5 minutes to 2 minutes.  
- Designed and launched marketing web page section, boosting site engagement by 15 %.  
- Developed and reorganized GitHub repository, cutting onboarding time 30 % and streamlining workflow.  
- Optimized site performance and UX, aligning with SEO best practices, increasing traffic 10 %.

In [74]:
# Get gemma4b to answer  # Comment describing which model you’re running.

response = ollama.chat.completions.create(  # Calls model.
    model=MODEL_GEMMA4,  # Selects gemma3:4b.
    messages=prompts["Few-Shot"]  # Uses few-shot prompt variant.
)  # End API call.

reply = response.choices[0].message.content  # Extracts reply text.
display(Markdown(reply))  # Displays reply in Markdown.

Here are the rewritten bullet points based on your guidelines:

*   Developed a streamlined payment process by integrating Google Forms, reducing transaction time from 5 to 2 minutes.
*   Designed and deployed impactful marketing web pages, enhancing user engagement and brand visibility.
*   Streamlined workflow by developing and organizing a robust Github repository for collaborative software development.
*   Led technical initiatives resulting in a 2-minute reduction in payment processing, improving user experience.

In [77]:
# Get gemma to answer  # Comment describing which model you’re running.

response = ollama.chat.completions.create(  # Calls model.
    model=MODEL_GEMMA,  # Selects gemma3:1b.
    messages=prompts["Role-Play (Recruiter)"]  # Uses recruiter role-play variant.
)  # End API call.

reply = response.choices[0].message.content  # Extracts reply text.
display(Markdown(reply))  # Displays reply in Markdown.

Okay, here are four rewritten bullet points for each of your input points, formatted for ATS and aiming for impactful language, incorporating relevant industry keywords and measurable achievements. I’ve focused on clarity and brevity.

**1. Integrated the Google Form Transaction page with the website, resulting in a 2-minute payment process reduction from 5 minutes.**

*   **Implemented** the Google Form Transaction page seamlessly into the website architecture, shaving 2 minutes off the payment process.
*   **Executed** an integration project, streamlining customer flow and reducing payment completion time by 2 minutes.
*   **Modernized** the checkout process with the Google Form integration, enhancing user experience and improving conversion rates.
*   **Achieved** a 2-minute time savings through an optimized checkout flow, a tangible benefit for both customers and the business.


**2. Designed and deployed the marketing web page section of the site, enhancing user engagement.**

*   **Conceptualized** a compelling marketing page design, boosting user engagement by 15% through visual strategy.
*   **Architected** a user-friendly marketing section, improving site navigation and comprehension speeds via a tailored design.
*   **Launched** the marketing section, optimizing user experience and increasing web page impact by 10% due to responsive design.
*   **Developed** a visually appealing and functional marketing page, driving traffic and lead generation for business goals.

**3. Developed and organized the Github repository to streamline workflow, boosting code collaboration.**

*   **Established** a robust Github repository for code management, fostering team collaboration and project efficiency.
*   **Implemented** workflow automation within the Github repository, reducing code review and delivery time approximately 20% .
*   **Streamlined** development processes utilizing the Github repository, increasing overall team productivity.
*   **Optimized** codebase architecture through Github collaboration, improving code quality and maintenance speed.


**4. Integrated the Google Form Transaction page with the website, resulting in 2 min payment process reduction from 5 minutes.**

*   **Executed** a critical integration, reducing the transaction process by 2 minutes to save valuable time.
*   **Collaborated** on the complete infrastructure, integrating Google Form transaction to improve checkout process speed.
*   **Improved** customer experience with the integrated system, a measurable improvement in 2 min change.
*   **Achieved** a demonstrable improvement in checkout time with the integration, boosting conversion rates


---

**To help me refine these further and tailor them even more precisely, could you tell me:**

*   **What industry is this work for?** (e.g., e-commerce, SaaS, financial services, etc.)
*   **What is the overall goal of this project?** (e.g., increase sales, improve user engagement, reduce costs)

In [67]:
# Get deepseek to answer  # Comment describing which model you’re running.

response = ollama.chat.completions.create(  # Calls model.
    model=MODEL_DEEPSEEK,  # Selects deepseek-r1:1.5b.
    messages=prompts["Chain-of-Thought"]  # Uses CoT variant again.
)  # End API call.

reply = response.choices[0].message.content  # Extracts reply text.
display(Markdown(reply))  # Displays reply in Markdown.

- **Integrated**: Successfully linked the Google Form page with the website, streamlining user experiences and reducing overall payment time.
**From 5 minutes to 2 minutes**
  
- **Dedicated**: Effectively developed and deployed a marketing section of the site aimed at enhancing conversion rates.
**Developed and deployed**: To boost conversion rates, while streamlining interactions.

  
- **Streamlined**: Optimized the GitHub repository efforts to streamline workflows and improve collaboration.
**Reduced bug resolution time**, while fostering clearer documentation and better communication.

  
- **Improved API Integration**: After reviewing an old log file, enhances backend transaction processing speed.
**Improved from: Real-time transaction**, real-time matching.

In [68]:
# Get llama to answer  # Comment describing which model you’re running.

response = ollama.chat.completions.create(  # Calls model.
    model=MODEL_LLAMA,  # Selects llama3.2.
    messages=prompts["Zero-Shot"]  # Uses role-play variant.
)  # End API call.

reply = response.choices[0].message.content  # Extracts reply text.
display(Markdown(reply))  # Displays reply in Markdown.


# llama 3.2 is traained on less parameters due to which it's bullet point are not seperated in new lines.

Here are three sets of rewritten bullet points for each input:

**Bullet Point 1:**

Original:
Integrated the Google Form Transaction page with the website which lead to 2 min payment process, from 5min 

Rewritten Bullet Points (3):

• Dramatically reduced checkout time by integrating Google Form Transaction page, shaving off 80% of processing time.
• Streamlined payment process, slashing average transaction completion time to 2 minutes.
• Enhanced user experience with seamless integration, leading to a significant increase in conversion rates.

**Bullet Point 2:**

Original:
Designed and deployed the marketing web page section of the site.  

Rewritten Bullet Points (3):

• Spearheaded the design and deployment of key marketing webpage elements, resulting in a 25% boost in engagement metrics.
• Conceptualized and brought to life a visually appealing and user-friendly marketing platform that exceeded client expectations.
• Effectively enhanced website's CTA conversion rates by 15% through effective marketing page design and layout.

**Bullet Point 3:**

Original:
Developed and organized the Github repository to streamline workflow. 

Rewritten Bullet Points (3):

• Successfully reorganized GitHub repository, significantly reducing branch conflict resolution time by 90%.
• Implemented efficient collaboration tools, fostering a culture of transparency and timely issue management within the team.
• Streamlined code review process, cutting average code review time in half and improving overall developer satisfaction.

In [69]:
# (empty cell)  # This cell intentionally has no executable code.# (empty cell)  # This cell intentionally has no executable code.

In [70]:
''' 
Make sure to check v2_comment variant as well.

'''

' \nMake sure to check v2_comment variant as well.\n\n'

In [ ]:
"""
 GPT-OSS:120b and gemma3:4b performs relatively good as an open source LLM models.
 llama 3.2 has relatively below average performance as compare to other.
 Prompting Technique:- Tried to use "policy style" approach.
                       Implemented very simple words so that open source light LLMs can easily interpret.
"""